In [1]:
import pandas as pd
import numpy as np 

In [6]:
df = pd.read_csv("gabungannya_train_lr.csv")
df.head()

,ecg_id,channel-0,channel-1,channel-2,channel-3,channel-4,channel-5,channel-6,channel-7,channel-8,channel-9,channel-10,channel-11,NORM,MI,STTC,HYP,CD
0,1,-0.119,-0.055,0.064,0.086,-0.091,0.004,-0.069,-0.031,0.000,-0.026,-0.039,-0.079,1,0,0,0,0
1,1,-0.116,-0.051,0.065,0.083,-0.090,0.006,-0.064,-0.036,-0.003,-0.031,-0.034,-0.074,1,0,0,0,0
2,1,-0.120,-0.044,0.076,0.082,-0.098,0.016,-0.058,-0.034,-0.010,-0.028,-0.029,-0.069,1,0,0,0,0
3,1,-0.117,-0.038,0.080,0.077,-0.098,0.021,-0.050,-0.030,-0.015,-0.023,-0.022,-0.064,1,0,0,0,0
4,1,-0.103,-0.031,0.072,0.066,-0.087,0.021,-0.045,-0.027,-0.020,-0.019,-0.018,-0.058,1,0,0,0,0


### Ekstraksi Fitur Time Series
Karena setiap `ecg_id` memiliki 1000 baris observasi (time steps), kita tidak bisa langsung memasukkan matriks 2D tersebut per sampel ke algoritma Random Forest biasa. Kita perlu mengubahnya menjadi representasi fitur 1D per `ecg_id`.
Pendekatan yang dilakukan adalah mengekstraksi fitur statistik (mean, standard deviasi, max, min, median) dari setiap channel per `ecg_id`.

In [7]:
# Ekstraksi fitur statistik yang cepat menggunakan fungsi agg dari pandas
channels = [col for col in df.columns if 'channel' in col]
agg_funcs = ['mean', 'std', 'max', 'min', 'median']

# Membuat dictionary aggregasi untuk setiap channel
agg_dict = {ch: agg_funcs for ch in channels}

print("Mengekstraksi fitur statistik dari setiap channel...")
X_features = df.groupby('ecg_id').agg(agg_dict)

# Menggabungkan nama kolom MultiIndex hasil aggregasi agar menjadi 1 level (contoh: channel-0_mean)
X_features.columns = [f"{col[0]}_{col[1]}" for col in X_features.columns]

# Mengambil label untuk setiap ecg_id (karena label konstan untuk 1000 baris di ecg_id yang sama)
labels = ['NORM', 'MI', 'STTC', 'HYP', 'CD']
y_labels = df.groupby('ecg_id')[labels].first()

print("Bentuk X_features:", X_features.shape)
print("Bentuk y_labels:", y_labels.shape)

display(X_features.head())
display(y_labels.head())

Mengekstraksi fitur statistik dari setiap channel...
Bentuk X_features: (17418, 60)
Bentuk y_labels: (17418, 5)


,channel-0_mean,channel-0_std,channel-0_max,channel-0_min,channel-0_median,channel-1_mean,channel-1_std,channel-1_max,channel-1_min,channel-1_median,...,channel-10_mean,channel-10_std,channel-10_max,channel-10_min,channel-10_median,channel-11_mean,channel-11_std,channel-11_max,channel-11_min,channel-11_median
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,0.001508,0.109077,0.706,-0.195,-0.0090,0.000723,0.083265,0.435,-0.140,-0.0235,...,-0.000557,0.089249,0.530,-0.097,-0.0300,0.000832,0.101990,0.602,-0.155,-0.0110
2,-0.002181,0.133495,0.664,-0.349,-0.0430,-0.000165,0.249870,1.529,-0.419,-0.0150,...,0.006375,0.228789,1.839,-0.492,-0.0395,0.003389,0.186720,1.394,-0.204,-0.0215
3,-0.002770,0.121260,0.827,-0.164,-0.0370,0.005145,0.144462,0.715,-0.288,-0.0030,...,0.005362,0.110702,0.851,-0.175,-0.0140,0.004239,0.100892,0.705,-0.128,-0.0130
4,-0.003264,0.137871,0.503,-0.576,-0.0265,0.003502,0.438380,1.752,-1.284,-0.0060,...,-0.007380,0.394892,2.221,-1.269,-0.0990,-0.004061,0.298061,1.639,-0.796,-0.0790
5,-0.003769,0.082748,0.440,-0.267,-0.0230,-0.003054,0.236899,1.293,-0.619,-0.0230,...,0.000208,0.211009,1.493,-0.187,-0.0510,-0.000053,0.180299,1.267,-0.151,-0.0440


,NORM,MI,STTC,HYP,CD
ecg_id,,,,,
1,1,0,0,0,0
2,1,0,0,0,0
3,1,0,0,0,0
4,1,0,0,0,0
5,1,0,0,0,0


### Pelatihan Model Random Forest
Melakukan Multi-label Classification menggunakan `RandomForestClassifier`.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Membagi dataset menjadi training dan testing
# Catatan: Karena pada tahap ini data yang di-load baru data "test", di sini kita contohkan split secara lokal.
# Jika Anda ingin melatih model pada `gabungannya_train_lr.csv` secara penuh, load data tersebut lalu 
# jalankan ekstraksi fitur .agg() yang sama seperti di atas untuk mendapatkan X_train penuh.
X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels, test_size=0.2, random_state=42)

# Inisialisasi model Random Forest
# RF di scikit-learn sudah mendukung multilabel classification secara langsung.
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("Melatih model Random Forest...")
rf_model.fit(X_train, y_train)
print("Model selesai dilatih!")

Melatih model Random Forest...
Model selesai dilatih!


### Evaluasi Model
Kita evaluasi model dengan melihat metriks accuracy dan F1-Score untuk multilabel.

In [9]:
# Melakukan prediksi ke data uji
y_pred = rf_model.predict(X_test)

print("Subset Accuracy (Exact Match):", accuracy_score(y_test, y_pred))
print("Macro F1-Score:", f1_score(y_test, y_pred, average='macro', zero_division=0))
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=labels, zero_division=0))

Subset Accuracy (Exact Match): 0.42738231917336394
Macro F1-Score: 0.5041117678321501

--- Classification Report ---
              precision    recall  f1-score   support

        NORM       0.79      0.72      0.75      1556
          MI       0.74      0.29      0.42       878
        STTC       0.75      0.22      0.34       824
         HYP       0.77      0.30      0.44       405
          CD       0.85      0.43      0.57       779

   micro avg       0.79      0.45      0.58      4442
   macro avg       0.78      0.39      0.50      4442
weighted avg       0.78      0.45      0.55      4442
 samples avg       0.52      0.48      0.49      4442

